# Example 06 - Uncertainty Propagation on a 1D FES

This notebook mirrors `examples/06_uncertainty.py`. It builds a
synthetic 1D double-well free-energy surface, perturbs it with
uncertainty models, and estimates confidence intervals on rates and
exit times via bootstrap resampling.


In [ ]:
from pathlib import Path
import sys


def find_repo_root(start=None):
    current = (start or Path.cwd()).resolve()
    for candidate in [current] + list(current.parents):
        if (candidate / "pyproject.toml").exists() and (candidate / "stochkin").is_dir():
            return candidate
    raise RuntimeError("Could not locate the repository root from the current working directory.")


ROOT = find_repo_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

DATA_DIR = ROOT / "examples" / "data"
OUT_DIR = ROOT / "notebooks" / "output"
OUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Repository root: {ROOT}")
print(f"Notebook output directory: {OUT_DIR}")


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import stochkin as sk

from stochkin.plotting import _apply_pub_axes
from stochkin.style import publication_style


In [ ]:
def make_synthetic_fes(n=300, barrier=8.0):
    s = np.linspace(0.0, 1.0, n)
    x = 2.0 * s - 1.0
    F = barrier * (1.0 - x ** 2) ** 2
    F -= F.min()
    return s, F


def make_synthetic_D(s, D0=0.02, amp=0.005):
    return D0 - amp * np.exp(-((s - 0.5) ** 2) / (2 * 0.05 ** 2))


In [ ]:
n_bootstrap = 200
seed = 42

s, F = make_synthetic_fes()
D = make_synthetic_D(s)

res1 = sk.bootstrap_ctmc_1d(
    s,
    F,
    D,
    F_err=0.5,
    D_rel_err=0.3,
    n_bootstrap=n_bootstrap,
    seed=seed,
    T=300.0,
    time_unit="ps",
    verbose=True,
)

F_sigma = 0.3 + 0.8 * np.exp(-((s - 0.5) ** 2) / (2 * 0.08 ** 2))
D_lo = D * np.exp(-0.5)
D_hi = D * np.exp(+0.5)

res2 = sk.bootstrap_ctmc_1d(
    s,
    F,
    D,
    F_err=F_sigma,
    D_lo=D_lo,
    D_hi=D_hi,
    n_bootstrap=n_bootstrap,
    seed=seed + 1,
    T=300.0,
    time_unit="ps",
    verbose=True,
)


In [ ]:
print("Scenario 1")
print(res1.summary("ps"))
print()
print("Scenario 2")
print(res2.summary("ps"))


In [ ]:
output_path = OUT_DIR / "06_uncertainty.png"

with publication_style():
    fig, axes = plt.subplots(
        2,
        2,
        figsize=(9, 6.5),
        gridspec_kw={"hspace": 0.45, "wspace": 0.38},
    )

    ax = axes[0, 0]
    ax.fill_between(s, F - F_sigma, F + F_sigma, alpha=0.25, color="C0", label="sigma_F(s)")
    ax.plot(s, F, "C0-", lw=1.5, label="F(s)")
    ax.set_xlabel("s")
    ax.set_ylabel("F [kJ/mol]")
    ax.set_title("(a) FES +/- uncertainty")
    ax.legend(fontsize=8)
    _apply_pub_axes(ax)

    ax = axes[0, 1]
    ax.fill_between(s, D_lo, D_hi, alpha=0.25, color="C1", label="D CI")
    ax.plot(s, D, "C1-", lw=1.5, label="D(s)")
    ax.set_xlabel("s")
    ax.set_ylabel("D [CV^2/ps]")
    ax.set_title("(b) Diffusion +/- CI")
    ax.legend(fontsize=8)
    _apply_pub_axes(ax)

    ax = axes[1, 0]
    k01 = res2.K_samples[:, 0, 1]
    ax.hist(k01, bins=30, density=True, alpha=0.7, color="C2", edgecolor="white", linewidth=0.5)
    ax.axvline(res2.K_mean[0, 1], color="k", ls="--", lw=1.2, label=f"mean = {res2.K_mean[0, 1]:.4g}")
    ax.axvline(res2.K_ci_lo[0, 1], color="grey", ls=":", lw=1)
    ax.axvline(res2.K_ci_hi[0, 1], color="grey", ls=":", lw=1, label=f"{res2.confidence_level:.0%} CI")
    ax.set_xlabel("k_0->1 [ps^-1]")
    ax.set_ylabel("density")
    ax.set_title("(c) Rate 0->1 bootstrap")
    ax.legend(fontsize=8)
    _apply_pub_axes(ax)

    ax = axes[1, 1]
    tau0 = res2.exit_mean_samples[:, 0]
    ax.hist(tau0, bins=30, density=True, alpha=0.7, color="C3", edgecolor="white", linewidth=0.5)
    ax.axvline(
        res2.exit_mean_mean[0],
        color="k",
        ls="--",
        lw=1.2,
        label=f"mean = {res2.exit_mean_mean[0]:.4g}",
    )
    ax.axvline(res2.exit_mean_ci_lo[0], color="grey", ls=":", lw=1)
    ax.axvline(
        res2.exit_mean_ci_hi[0],
        color="grey",
        ls=":",
        lw=1,
        label=f"{res2.confidence_level:.0%} CI",
    )
    ax.set_xlabel("<tau_exit> [ps]")
    ax.set_ylabel("density")
    ax.set_title("(d) Exit time basin 0 bootstrap")
    ax.legend(fontsize=8)
    _apply_pub_axes(ax)

    fig.savefig(output_path, dpi=300, bbox_inches="tight")

print(f"Saved {output_path.relative_to(ROOT)}")
plt.show()
